In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

import h5py
import matplotlib.pyplot as plt

import tempfile
import os
import atexit
import glob
import shutil
from pathlib import Path

from dotenv import load_dotenv

from models.resnet_rnn import ResnetRNN
from models.resnet_rnn_stated import ResnetRNNStated


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

print("This can be run after preprocessing.")

In [ ]:
load_dotenv("config.env")

# --- Dataset parameters ---
USE_DVS = int(os.getenv("USE_DVS"))
USE_APS = int(os.getenv("USE_APS"))
MODEL_DIMS = USE_DVS + USE_APS
if MODEL_DIMS == 0: raise Exception("MODEL_DIMS must be non-zero")
PRETRAINED = bool(int(os.getenv("PRETRAINED")))
BINSIZE_MS = int(os.getenv("BINSIZE_MS"))
HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"
MODEL_TYPE = os.getenv("MODEL_TYPE")

BATCH_SIZE = int(os.getenv("BATCH_SIZE"))

# --- Location parameters ---
MODEL_SAVE_DIR = os.getenv("MODEL_SAVE_DIR")
DATASET_DIR = os.getenv("OUT_DIR")
TRAINING_DIR = f"{DATASET_DIR}/Training"
TEST_DIR = f"{DATASET_DIR}/Test"

In [ ]:
# TODO: Set the images dir
IMAGES_DIR = "/srv/scratch/A2RoboRes/OltansDatasets/img/CNNLSTM"
# SETUP MODEL
if MODEL_TYPE == "Resnet":
    model = models.resnet34(pretrained=PRETRAINED)
    model.conv1 = nn.Conv2d(in_channels=MODEL_DIMS, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)
    model.to(device)

elif MODEL_TYPE == "ResnetRNN":
    model = ResnetRNN(hidden_dim=128, num_layers=1, in_channels=MODEL_DIMS, pretrained=PRETRAINED).to(device)

elif MODEL_TYPE == "ResnetRNNStated":
    model = ResnetRNNStated(hidden_dim=128, num_layers=1, in_channels=MODEL_DIMS, pretrained=PRETRAINED).to(device)

else:
    raise Exception(f"There is no defined model for {MODEL_TYPE}!")

In [ ]:
model_files = glob.glob(MODEL_SAVE_DIR + "/*.pth")
best_model_dir = f"{MODEL_SAVE_DIR}/best/*.pth"
last_model_dir = f"{MODEL_SAVE_DIR}/last/*.pth"

model_files = sorted(model_files, key=os.path.getctime)

# pull best and last from a copy from tmpdir, in case they change while plotting (i.e. training is going on in parallel)
tmpdir = tempfile.mkdtemp()
atexit.register(lambda: shutil.rmtree(tmpdir, ignore_errors=True))
for path in glob.glob(best_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
for path in glob.glob(last_model_dir):
    shutil.copy2(path, os.path.join(tmpdir, os.path.basename(path)))
    # first best, then last
    model_files.extend(sorted(glob.glob(tmpdir + "/*.pth"), key=os.path.getctime))

train_files = sorted(glob.glob(TRAINING_DIR + "/*.hdf5"))
test_files = sorted(glob.glob(TEST_DIR + "/*.hdf5"))
whole_files = sorted(glob.glob(DATASET_DIR + "/*.hdf5"))

In [ ]:
# Delete old images
try:
        os.mkdir(IMAGES_DIR)
except FileExistsError:
    for img in glob.glob(IMAGES_DIR + "/*"):
        os.remove(img)

In [ ]:
criterion = nn.MSELoss()
steering_slope = 1 / 180  # (output_max - output_min)/(input_max - input_min) = (1-(-1))/(180-(-180))
inv_steering_slope = 180

def generate_predictions(model, file_path, batch_size=BATCH_SIZE):
    inv_apsdvs_max = 1 / 255
    def normalise(data):
            # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
            new_data = np.empty_like(data, dtype=DTYPE)
            new_data = (data * inv_apsdvs_max).astype(DTYPE)
            return new_data

    # used for getting the data out at the end.
    def denormalise_steering(data):
        return data * inv_steering_slope
    
    model.eval()

    with h5py.File(file_path, "r") as f:
        dvs_frame = np.array(f[DVS_FRAME_NAME])
        data_len = dvs_frame.shape[0]
        aps_frame = np.array(f[APS_FRAME_NAME])
        steering_angles = np.array(f["steering_wheel_angle"])
        if USE_APS and USE_DVS:
            # we have batch x 224 x 224 x 2 after stack
            combined_frames = np.stack((dvs_frame, aps_frame), axis=3)
        else:
            combined_frames = aps_frame if USE_APS else dvs_frame
            combined_frames = np.expand_dims(combined_frames, axis=3)
        

        preds, gts = [], []
        for start in range(0, data_len, batch_size):
            end = min(start + batch_size, data_len)
            batch_data = combined_frames[start:end]
            batch_data = normalise(batch_data)
            # b, h, w, c = batch_data.shape
            # transformed_batch = np.empty((b, c, h, w), dtype=DTYPE)

            # shuffle dimensions
            batch_data = np.transpose(batch_data, (0, 3, 1, 2))
            if MODEL_TYPE in ["ResnetRNN", "ResnetRNNStated"]:
                    # create batch dimension
                    batch_data = np.expand_dims(batch_data, axis=0)
            batch_data = torch.from_numpy(batch_data).to(device)
            
            with torch.no_grad():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    batch_out = model(batch_data).cpu().numpy()
                    if MODEL_TYPE in ["ResnetRNN", "ResnetRNNStated"]:
                        # uncreate batch dimension
                        batch_out = np.squeeze(batch_out, axis=0)

                batch_out = denormalise_steering(batch_out)
                preds.append(batch_out)
                gts.append(steering_angles[start:end])

        preds = np.concatenate(preds, axis=0)
        gts = np.concatenate(gts, axis=0)
        # Convert to torch tensors
        gts = np.expand_dims(gts, axis=1)
        preds_tensor = torch.tensor(preds).to(device)
        gts_tensor = torch.tensor(gts).to(device)

        # Calculate loss
        loss = criterion(preds_tensor, gts_tensor)

        return preds, loss

In [ ]:
def plot_predictions(model, files, model_files, dataset_name):
    fig = plt.figure(figsize=(15, 7.5 * len(files)))
    for i, f in enumerate(files):
        announce_str = f"Working on {f}, File [{i + 1}/{len(files)}]"
        print(f"{announce_str}\n{'-' * len(announce_str)}")

        ax = plt.subplot(len(files), 1, i + 1)
        ax.set_title(f, fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel("Predicted Values", fontsize=12)
        ax.grid(True)
        ax.set_ylim(-90, 90)
        
        with h5py.File(f, "r") as f5:
            gt = np.array(f5["steering_wheel_angle"])
        time_axis = [x * (BINSIZE_MS / 1000) for x in range(len(gt))]
        ax.plot(time_axis, gt, label="truth", linewidth=2, linestyle='--')
        
        for model_file in model_files:
            epoch = Path(model_file).stem.split('_')[0]
            epoch = epoch.split('/')[-1][1:]
            model_dict = torch.load(model_file, weights_only=True, map_location=device)
            model.load_state_dict(model_dict["model_state_dict"])
            model.to(device)
            
            preds, loss = generate_predictions(model, f)
            print(f"RMSE for model {epoch} is {loss ** 0.5:.2f}")
            ax.plot(time_axis, preds, label=f"E:{epoch}, L:{loss ** 0.5:.2f}")

        l = ax.legend(loc='best')
        if len(model_files) == 0:
            print("No models found! just plotting ground truth.")
        
        else:
            # add the "best" and "last" to legend
            # edge case, best and last are the same
            if len(l.get_texts()) > 2:
                current_text = l.get_texts()[-2].get_text()
                l.get_texts()[-2].set_text(f"Best: {current_text}")
                current_text = l.get_texts()[-1].get_text()
                l.get_texts()[-1].set_text(f"Last: {current_text}")
            else:
                current_text = l.get_texts()[-1].get_text()
                l.get_texts()[-1].set_text(f"Best/Last: {current_text}")

    plt.tight_layout()
    svg_path = os.path.join(IMAGES_DIR, f"{dataset_name}.svg")
    fig.savefig(svg_path, format="svg")
    print(f"Saved plot to {svg_path}")
    plt.show()
    plt.close(fig)

In [ ]:
plot_predictions(model, test_files, model_files, "Test")

In [ ]:
plot_predictions(model, train_files, model_files, "Train")

In [ ]:
plot_predictions(model, whole_files, model_files, "Unedited")